# Data Audit

Audit WRDS schema resolution, extract manifests, sample coverage, and the results inventory.

In [ ]:
from pathlib import Path
import json

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def read_json(path):
    return json.loads((ROOT / path).read_text(encoding="utf-8"))


def maybe_read_csv(path, **kwargs):
    p = ROOT / path
    if not p.exists():
        return pd.DataFrame()
    return pd.read_csv(p, **kwargs)

ROOT

## Schema and Extract Manifests

These summaries should be enough to verify the data stack without exposing raw vendor rows.

In [ ]:
schema = read_json("artifacts/manifests/schema_audit.json")
full_extract = read_json("artifacts/manifests/full_extract.json")
feature_store = read_json("artifacts/manifests/full_feature_store.json")
results = read_json("docs/assets/results_manifest.json")
{
    "schema_status": schema.get("status"),
    "extract_outputs": sorted(full_extract.get("outputs", {}).keys()),
    "feature_rows": feature_store.get("diagnostics", {}).get("rows"),
    "results_status": results.get("status"),
}

## Year Shards

Missing 2026 rows are documented by manifests rather than filled or interpolated.

In [ ]:
manifest_dir = ROOT / "artifacts" / "manifests"
rows = []
for path in sorted(manifest_dir.glob("full_crsp_monthly_*.json")):
    if path.name == "full_crsp_monthly.json":
        continue
    obj = json.loads(path.read_text(encoding="utf-8"))
    rows.append({"manifest": path.stem, **obj.get("outputs", {})})
pd.DataFrame(rows).tail(10)

## Results Inventory

In [ ]:
print((ROOT / "docs/assets/results_inventory.md").read_text(encoding="utf-8"))